# 0. GPU 사용 변수 지정

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv() # .env 파일을 읽어와 환경 변수로 등록

client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HUGGINGFACE_API_KEY"],
)

# 실습

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_id = "distilbert-base-uncased-finetuned-sst-2-english"

# device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. 모델 및 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id).to(device)

# 2. 추론 수행
text = "I love this library!"
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.inference_mode():
    logits = model(**inputs).logits

# 3. 결과 해석 (Softmax를 통해 확률로 변환)
predictions = torch.softmax(logits, dim=-1)
label_id = torch.argmax(predictions).item()
label = model.config.id2label[label_id]

print(f"Result: {label} ({predictions[0][label_id]:.4f})")


## 실습 1 — 감성 분석 (영어, 기본 모델)

In [ ]:
from transformers import pipeline

# v5에서는 모델명을 직접 지정

# classifier = pipeline("text-classification")

classifier = pipeline(
    task="text-classification", 
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

result = classifier("This movie was absolutely fantastic!")
print(result)
# [{'label': 'POSITIVE', 'score': 0.9998}]


- 입력 여러개

In [ ]:
classifier(["I've been waiting for a HuggingFace course my whole life.", 
		"I hate this so much!"])

* 한글 버전

In [ ]:
from transformers import pipeline

kr_classifier = pipeline("text-classification", model="snunlp/KR-FinBert-SC")
kr_result = kr_classifier("엄마가 해준 밥을 안 먹으면 배고파요")
print(kr_result)

## 실습 2 — 감성 분석 (영어 금융)

In [ ]:
from transformers import pipeline

# 금융 도메인 특화 모델 지정
classifier = pipeline("text-classification", model="ProsusAI/finbert")

texts = [
    "The company reported record profits.",   # 긍정
    "The firm filed for bankruptcy.",          # 부정
    "Earnings were in line with estimates.",   # 중립
]

results = classifier(texts)
for text, result in zip(texts, results):
    print(f"입력: {text}")
    print(f"결과: {result['label']} ({result['score']:.4f})")
    print()

## 실습 3 — 감성 분석 (한국어 금융)

In [ ]:
from transformers import pipeline

# 한국어 금융 특화 모델
classifier = pipeline("text-classification", model="snunlp/KR-FinBert-SC")

texts = [
    "삼성전자가 역대 최대 실적을 기록하며 배당금을 대폭 상향했다.",
    "해당 기업은 대규모 손실 이후 법정관리를 신청했다.",
    "3분기 영업이익은 시장 컨센서스에 부합하는 수준이었다.",
]

results = classifier(texts)
for text, result in zip(texts, results):
    print(f"입력: {text}")
    print(f"결과: {result['label']} ({result['score']:.4f})")
    print()

## 실습 4 — 다국어 감성 분석 (5단계)

In [ ]:
from transformers import pipeline

# 다국어 지원, 5단계 감성 분류
classifier = pipeline(
    "sentiment-analysis",
    model="tabularisai/multilingual-sentiment-analysis"
)

result = classifier("잘난 척 멋진 척 폼 재던 나도 꽁꽁 얼어붙었다.")
print(result)
# [{'label': 'Positive', 'score': 0.87xx}]

## 실습 5 — 빈칸 채우기 (Fill-Mask)

In [ ]:
from transformers import pipeline

# [MASK] 위치에 올 단어를 예측 (BERT 사전학습 방식)

# klue/bert-base, snunlp/KR-FinBert
classifier = pipeline("fill-mask", "bert-base-multilingual-cased")
classifier_klue = pipeline("fill-mask", "klue/bert-base")
classifier_snulp = pipeline("fill-mask", "snunlp/KR-FinBert")

result = classifier("안녕하세요? 나는 [MASK] 모델입니다.")

print("-- result for basic model")
for r in result:
    print(f"예측 단어: {r['token_str']} | 확률: {r['score']:.4f}")
    print(f"문장: {r['sequence']}")
    print()


In [ ]:
result_klue = classifier_klue("안녕하세요? 나는 [MASK] 모델입니다.")

print("-- result for klue/bert-base model")
for r in result_klue:
    print(f"예측 단어: {r['token_str']} | 확률: {r['score']:.4f}")
    print(f"문장: {r['sequence']}")
    print()

In [ ]:
result_snulp = classifier_snulp("안녕하세요? 나는 [MASK] 모델입니다.")
print("-- result for snunlp/KR-FinBert")
for r in result_snulp:
    print(f"예측 단어: {r['token_str']} | 확률: {r['score']:.4f}")
    print(f"문장: {r['sequence']}")
    print()

## 실습 6 — 질의 응답 (QA)
⚠️ Transformers 5.x에서 pipeline("question-answering") 제거 → 직접 클래스 호출

- logits → softmax → 확률값

In [ ]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
import torch

model_name = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# 질문과 지문 정의
qa_samples = [
    {
        "question": "What is the capital of France?",
        "context": "France is a country in Western Europe. Its capital is Paris."
    },
    {
        "question": "Who invented the telephone?",
        "context": "The telephone was invented by Alexander Graham Bell in 1876."
    },
    {
        "question": "What language does Python use for indentation?",
        "context": "Python uses whitespace indentation to define code blocks."
    },
]

for sample in qa_samples:
    inputs = tokenizer(sample["question"], sample["context"], return_tensors="pt")

    # torch.no_grad() : 추론단계로 그래디언트(Gradient, 기울기) 계산을 비활성화
    with torch.no_grad():
        outputs = model(**inputs)

    # 정답 시작/끝 위치 추출
    start = torch.argmax(outputs.start_logits)
    end   = torch.argmax(outputs.end_logits) + 1
    
    # logits → softmax → 확률값
    start_prob = torch.softmax(outputs.start_logits, dim=-1)
    end_prob   = torch.softmax(outputs.end_logits,   dim=-1)

    # 각 최댓값이 해당 위치의 확률 = score
    start_score = torch.max(start_prob).item()
    end_score   = torch.max(end_prob).item()

    # 최종 score = start * end 확률의 곱
    score = start_score * end_score
    
    # answer = tokenizer.convert_tokens_to_string(
    #     tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][start:end])
    # )
    
    answer = tokenizer.decode(inputs.input_ids[0][start : end])

    print(f"Q: {sample['question']}")
    print(f"A: {answer}")
    print(f"Score  : {score:.4f}")
    print()

## 실습 7 — 번역 (한국어 → 영어)

⚠️ Transformers 5.x에서 `pipeline("translation")` 제거 → MarianMTModel 직접 사용

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-ko-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

sentences = [
    "안녕하세요, 저는 인공지능을 공부하고 있습니다.",
    "오늘 날씨가 정말 좋네요.",
    "한국의 경제 성장률이 올해 2%를 기록했습니다.",
]

# 배치 번역
inputs = tokenizer(sentences, return_tensors="pt", padding=True)
translated = model.generate(**inputs)
results = tokenizer.batch_decode(translated, skip_special_tokens=True)

for src, tgt in zip(sentences, results):
    print(f"입력: {src}")
    print(f"번역: {tgt}")
    print()

* 영->한 번역 : 실패

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

model_name_ko = "mradermacher/English-to-korean-translator-GGUF"
tokenizer_2 = MarianTokenizer.from_pretrained(model_name_ko)
model_2 = MarianMTModel.from_pretrained(model_name_ko)

sentences_2 = [
    "Hi, I'm studying AI.",
    "It's a great day.",
    "South Korea's economic growth rate is 2% this year.",
]

# 배치 번역
inputs_2 = tokenizer_2(sentences_2, return_tensors="pt", padding=True)
translated_2 = model_2.generate(**inputs_2)
results_2 = tokenizer_2.batch_decode(translated_2, skip_special_tokens=True)

for src, tgt in zip(sentences_2, results_2):
    print(f"입력: {src}")
    print(f"번역: {tgt}")
    print()

## 실습 8 — 요약 (영어)

> ⚠️ Transformers 5.x에서 `pipeline("summarization")` 제거 → 직접 클래스 호출

In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer
import torch

model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

article = """
NASA's Perseverance rover has discovered organic molecules on Mars.
The finding suggests that the red planet may have once harbored conditions
suitable for life. The rover collected rock samples from the Jezero Crater,
an ancient lake bed, where the molecules were detected.
"""

inputs = tokenizer(article, return_tensors="pt", max_length=1024, truncation=True)

with torch.no_grad():
    summary_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=100,
        num_beams=4,          # 빔 서치로 품질 향상
        early_stopping=True
    )

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(f"원문: {article.strip()}")
summary = summary.replace('.', '. \n')
print(f"요약:\n {summary}")

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_id = "google/pegasus-xsum"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    dtype=torch.float16,  # 최신 버전에서는 dtype이 표준입니다!
    device_map="auto",
    attn_implementation="sdpa"
)

# input_text = """Plants are remarkable organisms that produce their own food using a method called photosynthesis. This process involves converting sunlight, carbon dioxide, and water into glucose, which provides energy for growth. Plants play a crucial role in sustaining life on Earth by generating oxygen and serving as the foundation of most ecosystems."""

input_text = """We will build a Great Iron Dome missile defense shield over our entire country, a dome the likes of which the world has never seen before. We will restore the republic, and we will usher in the rich and wonderful future that our people so truly deserve. The discord and division in our society must be healed. We must heal it quickly. As Americans, we are bound together by a single fate and a shared destiny. We rise together, or we fall apart. I am running to be president for all of America, not half of America, because there is no victory in winning for half of America."""


input_ids = tokenizer(input_text, return_tensors="pt").to(model.device)

# static 캐시 오류가 날 경우 이 부분을 기본값으로 사용하세요
# output = model.generate(**input_ids, max_length=64)

output = model.generate(
    **input_ids,
    max_length=64,
    num_beams=8,               # 8개의 경로를 탐색해 최적의 문장 생성
    length_penalty=0.8,        # 요약이 너무 길어지지 않게 조절
    no_repeat_ngram_size=3,    # 3단어 이상 겹치는 구절 방지
    forced_eos_token_id=tokenizer.eos_token_id # 문장이 확실히 끝나도록 유도
)

print(tokenizer.decode(output[0], skip_special_tokens=True))


## 실습 9 — 임베딩 (Sentence Embedding)

In [ ]:
from sentence_transformers import SentenceTransformer
from torch.nn.functional import cosine_similarity

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = [
    "The weather is lovely today.",   # 날씨 문장 1
    "It's so sunny outside!",          # 날씨 문장 2
    "He drove to the stadium.",        # 관계없는 문장
]

# 문장 → 384차원 벡터로 변환
embeddings = model.encode(sentences, convert_to_tensor=True)
print(f"임베딩 shape: {embeddings.shape}")  # (3, 384)
print()

# 문장 간 코사인 유사도 계산
pairs = [(0, 1), (0, 2), (1, 2)]
for i, j in pairs:
    score = cosine_similarity(
        embeddings[i].unsqueeze(0),
        embeddings[j].unsqueeze(0)
    ).item()
    print(f"[{sentences[i]}]")
    print(f"[{sentences[j]}]")
    print(f"유사도: {score:.4f}")
    print()

## 실습 10 — 텍스트 생성 (소형 LLM)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"사용 디바이스: {device}")

def generate(prompt, do_sample=False, num_beams=1, temperature=1.0):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=do_sample,
            num_beams=num_beams,
            temperature=temperature,
        )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

prompt = "머신러닝을 한 문장으로 설명하면?"

print(f"Q: {prompt}\n")
print(f"[탐욕 탐색]\n{generate(prompt)}\n")
print(f"[빔 서치]\n{generate(prompt, num_beams=4)}\n")
print(f"[샘플링]\n{generate(prompt, do_sample=True, temperature=0.7)}\n")

## 실습 11 — 파인튜닝 (IMDB 감성 분류)

### Step 1 — 데이터 로드

In [ ]:
from datasets import load_dataset

dataset = load_dataset('imdb')
print(dataset)

### Step 2 — 토크나이징

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512,
    )

# 전체 데이터셋 토크나이징
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 실습용 소규모 서브셋 (학습 시간 단축)
small_train = tokenized_datasets["train"].shuffle(seed=42).select(range(2000))
small_test  = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

print(f"학습: {len(small_train)}개 / 테스트: {len(small_test)}개")

### Step 3 — 모델 로드

In [ ]:
from transformers import AutoModelForSequenceClassification

# DistilBERT 위에 분류 헤드(Linear layer) 자동 추가
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2  # NEGATIVE(0) / POSITIVE(1)
)

> 💡 **경고 메시지 정상**: `classifier.weight` 새로 초기화 → 파인튜닝으로 학습할 부분

### Step 4 — 학습

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1"      : f1_score(labels, predictions, average="weighted"),
    }

tr_args = TrainingArguments(
    output_dir                  = "./results",
    learning_rate               = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    num_train_epochs            = 3,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",  # ⚠️ 5.x: evaluation_strategy → eval_strategy
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    fp16                        = True,     # GPU 혼합 정밀도 (속도 향상)
)

trainer = Trainer(
    model           = model,
    args            = tr_args,
    train_dataset   = small_train,
    eval_dataset    = small_test,
    compute_metrics = compute_metrics,
)

trainer.train()

### Step 5 — 평가

In [ ]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()
model.to(device)

# 불필요한 컬럼 제거
small_test_torch = small_test.remove_columns(
    [col for col in small_test.column_names
     if col not in ["input_ids", "attention_mask", "label"]]
)
small_test_torch = small_test_torch.rename_column("label", "labels")
small_test_torch.set_format("torch")

loader = DataLoader(small_test_torch, batch_size=16)
all_preds, all_labels = [], []

for batch in loader:
    input_ids      = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
    all_preds.extend(preds)
    all_labels.extend(batch["labels"].numpy())

print(f"Accuracy : {accuracy_score(all_labels, all_preds):.4f}")
print(f"F1 Score : {f1_score(all_labels, all_preds, average='weighted'):.4f}")

### Step 6 — 저장 및 추론

In [ ]:
# 모델 저장
trainer.save_model("./my-imdb-model")
tokenizer.save_pretrained("./my-imdb-model")
print("저장 완료 → ./my-imdb-model")

In [ ]:
# 저장된 모델로 추론
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_path = "./my-imdb-model"
inf_tokenizer = AutoTokenizer.from_pretrained(model_path)
inf_model = AutoModelForSequenceClassification.from_pretrained(model_path)

reviews = [
    "This movie was absolutely fantastic! I loved every minute of it.",
    "Terrible film. Complete waste of time and money.",
    "This movie is a miracle; it cured my cancer."
]

for review in reviews:
    inputs = inf_tokenizer(review, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = inf_model(**inputs)
    pred  = torch.argmax(outputs.logits, dim=-1).item()
    label = "POSITIVE" if pred == 1 else "NEGATIVE"
    print(f"리뷰: {(review[:50]+'...') if len(review) > 50 else review}")
    print(f"결과: {label}")
    print()